# EcoVolt Energy Consumption Prediction (LSTM)

This notebook implements a Deep Learning model (LSTM) to predict the next hour's electricity consumption based on the previous 24 hours of data. The model utilizes multivariate input including consumption and various production sources.

## 1. Environment Setup (Google Colab)
If you are running this notebook in Google Colab, run the cell below to upload your `dataset.csv` file.

In [ ]:
try:
    from google.colab import files
    print("Running in Google Colab. Please upload 'dataset.csv'.")
    uploaded = files.upload()
except ImportError:
    print("Running locally.")

## 2. Imports and Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

# Set seed for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

## 3. Data Loading and Preprocessing

**Explication (Attendue):**
- **Ordre Temporel Critique**: Contrairement aux datasets classiques, une série temporelle dépend de l'ordre. `t` dépend de `t-1`. Mélanger les données détruirait cette causalité.
- **Pas de Shuffle**: Si on mélange, on perd la continuité temporelle nécessaire pour que le modèle 'apprenne' la séquence.

In [ ]:
# Determine file path (handles both Colab upload and local folder)
if os.path.exists('dataset.csv'):
    file_path = 'dataset.csv'
elif os.path.exists('data/dataset.csv'):
    file_path = 'data/dataset.csv'
else:
    raise FileNotFoundError("dataset.csv not found. Please upload it or check the path.")

# Load the dataset
df = pd.read_csv(file_path)

# Convert DateTime to datetime objects
df['DateTime'] = pd.to_datetime(df['DateTime'])

# Sort by time to ensure correct sequential order
df = df.sort_values('DateTime')

# Set DateTime as index
df.set_index('DateTime', inplace=True)

# Display first few rows
print(df.head())
print(df.info())

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(df.index, df['Consumption'], label='Consumption')
plt.title('Hourly Energy Consumption over Time')
plt.xlabel('Date')
plt.ylabel('Consumption (MW)')
plt.legend()
plt.show()

## 5. Feature Scaling (Normalization)
**Explication (Attendue):**
- **Indispensable en Deep Learning**: Les LSTM sont sensibles à l'échelle des données. Des grandes valeurs (ex: 6000 MW) peuvent causer des gradients instables.
- **MinMaxScaler vs StandardScaler**: `MinMaxScaler` (0-1) est généralement préféré pour les réseaux de neurones (surtout avec tanh/sigmoid) pour éviter la saturation des gradients.

In [ ]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

# Check the shape of scaled data
print("Scaled data shape:", scaled_data.shape)

## 6. Creating Sequences (Windowing)
**Explication (Attendue):**
- **Format 3D (Sample, Timesteps, Features)**: Le LSTM traite des séquences, pas des points isolés.
- **Samples**: Nombre total d'exemples.
- **Timesteps**: Longueur de la séquence (ici 24 heures).
- **Features**: Nombre de variables observées à chaque pas de temps (9 variables).

In [ ]:
def create_sequences(data, window=24):
    X, y = [], []
    # We loop via the length of data minus the window size
    for i in range(window, len(data)):
        X.append(data[i-window:i])
        # The target is the 'Consumption' at the current step.
        # Assuming 'Consumption' is the first column (index 0)
        y.append(data[i, 0])
    return np.array(X), np.array(y)

# Create sequences
WINDOW_SIZE = 24
X, y = create_sequences(scaled_data, window=WINDOW_SIZE)

print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Train/Test Split
**Explication (Attendue):**
- **Pas de Random Split**: En séries temporelles, il est interdit de mélanger le futur et le passé (Data Leakage). On doit tester le modèle sur des données *futures* qu'il n'a jamais vues pendant l'entraînement.

In [ ]:
split_idx = int(0.8 * len(X))

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print("Training shape:", X_train.shape, y_train.shape)
print("Testing shape:", X_test.shape, y_test.shape)

## 8. Building the LSTM Model

In [ ]:
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

## 9. Model Training

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=32,
    verbose=1
)

## 10. Evaluation and Visualization

In [ ]:
# Plot Loss
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss During Training')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend()
plt.show()

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Visualization of results (Scaled)
plt.figure(figsize=(15, 6))
plt.plot(y_test, label='Real Consumption', alpha=0.7)
plt.plot(y_pred, label='Predicted Consumption', alpha=0.7)
plt.title('Energy Consumption Prediction (Test Set)')
plt.xlabel('Time Steps')
plt.ylabel('Normalized Consumption')
plt.legend()
plt.show()

In [ ]:
# Optional: Inverse Transform for Real Scale Visualization
# Create a dummy array to match the scaler's expected input shape
dummy_test = np.zeros((len(y_test), scaled_data.shape[1]))
dummy_pred = np.zeros((len(y_pred), scaled_data.shape[1]))

# Place the target variable in the first column (assuming Consumption is at index 0)
dummy_test[:, 0] = y_test
dummy_pred[:, 0] = y_pred.flatten()

# Inverse transform
y_test_original = scaler.inverse_transform(dummy_test)[:, 0]
y_pred_original = scaler.inverse_transform(dummy_pred)[:, 0]

plt.figure(figsize=(15, 6))
plt.plot(y_test_original, label='Real (MW)', alpha=0.7)
plt.plot(y_pred_original, label='Predicted (MW)', alpha=0.7)
plt.title('Energy Consumption: Real vs Predicted (MW)')
plt.xlabel('Time Steps')
plt.ylabel('Consumption (MW)')
plt.legend()
plt.show()